# Instagram Likes Prediction: Real-World Regression

**You've learned Linear Regression basics. Now apply it to real social media data!** 📱

**The Challenge:**
Predict how many likes an Instagram post will get based on account metrics.

**You'll learn:**
- Work with real-world messy data
- Engineer features from historical data
- Compare model performance
- Iterate to improve predictions

**Dataset:** 1,000+ Instagram posts with follower counts, timestamps, and engagement metrics

**Business value:** Content creators and marketers use this to forecast post performance!

Let's predict some likes! 👇

## Data Exploration

a) Download the [`posts.csv.bz`](https://drive.google.com/uc?export=download&id=1O8ey3uytjqzRUQXTnmXkiqGBoa6lne1C) and store it in your `data` directory for this challenge. One row of this dataset represents an Instagram post.

<details>
<summary markdown="span">Not sure where your downloaded file went?</summary>

You can move it manually into the `data` folder, or run one of these in your terminal:

- **macOS/Linux:** `mv ~/Downloads/posts.csv.bz data/`
- **Windows (WSL):** `mv /mnt/c/Users/YOUR_USERNAME/Downloads/posts.csv.bz data/`

</details>

**About this file:** This dataset is **compressed** using bz2 compression to save space. The `.bz` extension indicates bz2 compression. Pandas can read compressed files directly!

Save it to the variable `df_posts`

In [ ]:
# YOUR CODE HERE

b) Column `id` designates the author. How many **unique** authors do we have ?

In [ ]:
# YOUR CODE HERE

c) Sort `df_posts` by **ascending date**.

In [ ]:
# YOUR CODE HERE

d) This dataset contains several posts for the same author.

By using the pandas function `drop_duplicates()` with parameter `keep=last`, **keep only the last post** made by each author.

**Save the result back to `df_posts`** (overwrite the existing dataframe).

This should leave you with a dataframe with as many rows as unique authors!

In [ ]:
# YOUR CODE HERE

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'data_preprocessing',
    df_shape=df_posts.shape,
    unique_authors=df_posts['id'].nunique(),
    is_sorted=df_posts['ts'].is_monotonic_increasing
)
result.write()
print(result.check())

### 📊 Visualize the Relationship

e) Use Plotly to draw a scatter plot between likes and followers. Do you see any **correlation**?


In [ ]:
import plotly.express as px

fig = px.scatter(df_posts, "followers", "likes")
fig.show()

**What do you see?**

- Positive correlation: More followers = More likes (generally)
- Lots of scatter: Followers alone **don't tell the whole story**!

💡 Insight: This suggests followers **might** be a useful feature for prediction

Next: Let's build a model to quantify this relationship! 👇

## 🤖 Build Predictive Models

**Our Plan:**
1. **Model 1:** Predict likes using only follower count (baseline)
2. **Model 2:** Add historical likes feature (improvement!)
3. Compare performance

### Model 1: Baseline (Followers Only)


#### Preprocessing

a) Isolate the target and the feature!

For our first model, we'll start with the feature being the number of followers. The target, as you already know, is the number of likes!

**Assign X and y appropriately!**

In [ ]:
# YOUR CODE HERE

b) Remember what we said about the importance of separating data into train and test splits?

Split X and y into `X_train`, `X_test`, `y_train` and `y_test`.

A good rule of thumb, which we will follow here is **80% for training and 20% for testing**

Don't make your life difficult, use the `train_test_split` function!

In [ ]:
# YOUR CODE HERE

c) Now we need to scale! I know you're excited to apply your new modelling skills, but we first need to make sure the data is scaled!

🚨 **Remember:** Fit on training, transform both! 🚨

Let's use the standard scaler 👍

In [ ]:
# YOUR CODE HERE

### Linear Regression

d) Train a Linear Regression model called `model_1` that predicts likes (our y) based on followers (our X)! Remember to only train it on our **scaled training set**, not on our test set!

In [ ]:
# YOUR CODE HERE

### Evaluation

e) Calculate three metrics **on the training data** to get an initial assessment of model performance:

- **R² score:** How much variance does the model explain? (0-1, higher = better)  
- **MSE (Mean Squared Error):** Average squared prediction error (lower = better)  
- **MAE (Mean Absolute Error):** Average absolute prediction error in target (lower = better)

Do you think our model is performing well?

<details>
<summary><i>Hint</i></summary>

  Use `mean_squared_error` and `mean_absolute_error` of module `sklearn.metrics`.

</details>

In [ ]:
# YOUR CODE HERE

R² is close to 0, so the model doesn't look very good.

MSE is quite difficult to interpret, but MAE is much simpler. 

Our model has an error of 32 likes on average.

f) Now we need to evaluate our model on **test data** to see how it performs on unseen data!

This is the true measure of model performance - the test set simulates real-world predictions.

Use your model to predict likes for test data and store the predictions in `pred_model_1`.

In [ ]:
# YOUR CODE HERE

In [ ]:
from nbresult import ChallengeResult

# 🔥 SAVE model_1's test data before it gets overwritten
X_test_scaled_model_1 = X_test_scaled.copy()
y_test_model_1 = y_test.copy()

result = ChallengeResult(
    'model1',
    X_shape=X.shape,
    pred_length=len(pred_model_1),
    model_1_r2=float(model_1.score(X_test_scaled, y_test))
)
result.write()
print(result.check())

g) What is R² score value on test data? What is the MSE (mean squared error)?

Do you think our model is a good one?

In [ ]:
# YOUR CODE HERE

Results are similar, our model is not really good. It was expected because with such simple model, you cannot get very accurate results.

## 🔧 Feature Engineering: Can We Do Better?

Follower count alone isn't enough. What else predicts likes?

**Hypothesis:** Accounts with higher historical engagement will get more likes!

**New Feature:** `historical_likes` = Median likes from an author's PREVIOUS posts

### 📊 Calculate Historical Performance

**What we're building:**
- For each author, calculate median likes from their past posts
- Use this as a feature (accounts with high historical engagement likely get more likes!)

h) First let's load the data again, we want a completely fresh set with some of the information we dropped before. This time save the data into a variable called `archive`.

In [ ]:
# YOUR CODE HERE

i) Sort the values by ascending date, just like we did before

In [ ]:
# YOUR CODE HERE

j) OK, now we want to identify the most recent posts per author and we'll save that in a variable called `most_recent_posts`.

In [ ]:
# YOUR CODE HERE

k) Then we want to calculate the median likes per author from all previous posts. 

This has three parts:

1. Remove the most recent posts from the archive

    We need to exclude the most recent posts because we want to calculate historical performance from previous posts only.

    This is tricky, if you're not sure how to do it there's a hint here:

In [ ]:
# YOUR CODE HERE

2. Calculate median likes per author

    Now group by author (`id`) and calculate the median of likes from all their previous posts.

    **Why median?** More robust than mean - not affected by one viral post!

In [ ]:
# YOUR CODE HERE

3. Rename the column. Change the name `likes` to `historical_likes` to distinguish it from current post likes. There are many ways to do it, but why not try out `.rename()`

In [ ]:
# rename column likes by historical_likes
median_last_posts = median_last_posts.rename({'likes': 'historical_likes'}, axis=1)

l) Merge with `most_recent_posts`. 

Add the `historical_likes` column to our dataframe of `most_recent_posts`.

Call the final dataframe `df_posts_new`

In [ ]:
# merge this column to initial dataframe
df_posts_new = most_recent_posts.merge(median_last_posts, on="id")

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'feature_engineering',
    df_posts_new_shape=df_posts_new.shape,
    has_historical_likes='historical_likes' in df_posts_new.columns,
    historical_likes_nan_count=int(df_posts_new['historical_likes'].isna().sum())
)
result.write()
print(result.check())

Have a look at `df_posts_new` to check our new feature has successfully been added

In [ ]:
df_posts_new

## 🚀 Model 2: Adding Historical Performance

**Hypothesis:** Combining follower count + historical engagement = better predictions!

Let's see if this improves our R² score! 👇

### Define Features & Target

m) Let's start all over, but now X is **followers and historical likes**.

Split X and y!

In [ ]:
# YOUR CODE HERE

n) Split into train and test split!

In [ ]:
# YOUR CODE HERE

o) Scale!

In [ ]:
# YOUR CODE HERE

p) Time to train a new model!

Train a new linear regression called `model_2` with our new `X_train_scaled`!


In [ ]:
# YOUR CODE HERE

q) Calculate R², MSE and MAE on the test data. What do you think of this new model?

In [ ]:
# YOUR CODE HERE

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'model2',
    X_shape=X.shape,
    model_2_r2=float(model_2.score(X_test_scaled, y_test)),
    model_1_r2=float(model_1.score(X_test_scaled_model_1, y_test_model_1))  # ✅ Use saved data!
)
result.write()
print(result.check())

## 📊 Compare Both Models

Run this code to compare:

In [ ]:
# Model 1 metrics (using saved test data)
model_1_r2 = model_1.score(X_test_scaled_model_1, y_test_model_1)
model_1_pred = model_1.predict(X_test_scaled_model_1)
model_1_mae = mean_absolute_error(y_test_model_1, model_1_pred)

print("Model 1 (Followers only):")
print(f"  R² = {model_1_r2:.3f}")
print(f"  MAE = {model_1_mae:.1f} likes")

# Model 2 metrics (using current test data)
model_2_r2 = model_2.score(X_test_scaled, y_test)
model_2_pred = model_2.predict(X_test_scaled)
model_2_mae = mean_absolute_error(y_test, model_2_pred)

print("Model 2 (Followers + Historical):")
print(f"  R² = {model_2_r2:.3f}")
print(f"  MAE = {model_2_mae:.1f} likes")

improvement = ((model_2_r2 - model_1_r2) / model_1_r2) * 100
print(f"🎉 Improvement: {improvement:.1f}% better R²!")

**Congratulations! You've completed your first REAL-WORLD regression project!** 🎉

**What makes this different from Challenge Two?**
- Real messy data (not clean NBA stats)
- Feature engineering (created new features)
- Model iteration (compared two approaches)
- Business context (social media analytics)

You're ready for production-level ML work! 💪

### 🏆 Key Takeaways

1. **Start Simple:** Begin with a basic model to establish a baseline
1. **Feature Engineering is Powerful:** Adding the right features can dramatically improve performance
1. **Multiple Metrics Matter:** Don't rely on just one metric - use R², MSE, and MAE together
1. **Preprocessing is Critical:** Scaling, splitting, and cleaning data properly is non-negotiable
1. **Iterate and Improve:** Data science is an iterative process - always look for ways to enhance your model